# Mestres do XGH — Aula 14
## Git e GitHub: a máquina do tempo do seu código

**Shift + Enter** roda a célula.

Sábado passado você fez um script que roda sozinho. Hoje você vai parar de perder as versões dele —
e descobrir uma coisa sobre o git que custa dinheiro para quem não sabe.

> **commit é um ponto de retorno; o resto é coragem.**

---
# 0. A oficina de hoje

Git mexe em arquivo de verdade. Então **não vamos mexer nos seus arquivos**: a célula abaixo cria uma
pasta temporária com um repositório de brinquedo, e tudo o que a gente fizer acontece lá dentro.

Ela também cria duas funções: `git(...)`, que chama o `git` do sistema e **imprime** o comando e a
saída, e `git_quieto(...)`, que faz o mesmo mas **devolve** a saída como texto, para a gente usar
dentro do código. Repare em três detalhes delas:

- `-c user.name=... -c user.email=...` — o git exige saber quem está commitando. A gente passa na hora,
  para o notebook **não depender da configuração da sua máquina**. No seu projeto de verdade você
  configura uma vez e esquece (está no bloco 8).
- `-c commit.gpgsign=false` — desliga assinatura de commit, que algumas máquinas têm ligada.
- `GIT_AUTHOR_DATE` fixo — com a data travada, **o código do commit sai igual na máquina de todo mundo**.
  Assim o que aparece na sua tela é o mesmo que está no slide.

**Rode e confira que o git existe aí.** Se der erro de comando não encontrado, instale o git
(`git-scm.com`) e reinicie o VS Code. No Colab já vem instalado.

In [ ]:
import os, subprocess, tempfile, time, shutil, sys, random
from pathlib import Path

random.seed(14)

# --- onde tudo vai acontecer (pasta temporária, longe dos seus arquivos) ---
BASE = Path(tempfile.mkdtemp(prefix="xgh-aula14-"))
REPO = BASE / "organizador"
REPO.mkdir(parents=True, exist_ok=True)

IDENT = ["-c", "user.name=Turma XGH",
         "-c", "user.email=turma@xgh.local",
         "-c", "commit.gpgsign=false"]

AMBIENTE = dict(os.environ,
                GIT_AUTHOR_DATE="2026-11-14T10:20:00-03:00",
                GIT_COMMITTER_DATE="2026-11-14T10:20:00-03:00")

def git_quieto(*args, onde=None):
    """Roda um comando git e DEVOLVE a saída, sem imprimir nada."""
    r = subprocess.run(["git", *IDENT, *args],
                       capture_output=True, text=True,
                       cwd=str(onde or REPO), env=AMBIENTE)
    return (r.stdout + r.stderr).rstrip()

def git(*args, onde=None):
    """Roda um comando git e imprime o comando e a saída."""
    saida = git_quieto(*args, onde=onde)
    print("$ git " + " ".join(args))
    print(saida if saida else "(sem saída)")
    print()

# --- a pasta do horror: como a gente versiona hoje ---
HORROR = BASE / "pasta_do_horror"
HORROR.mkdir(exist_ok=True)
for nome in ["organiza.py", "organiza_v2.py", "organiza_v2_ok.py",
             "organiza_FINAL.py", "organiza_FINAL_agora_vai.py",
             "organiza_FINAL_agora_vai (1).py", "organiza_backup_ontem.py"]:
    (HORROR / nome).write_text("# ...\n", encoding="utf-8")

# --- o script da Aula 13, versão 1 ---
ORGANIZA_V1 = '''import sys, shutil
from pathlib import Path

def organizar(pasta):
    pasta = Path(pasta)
    for arquivo in pasta.iterdir():
        if arquivo.is_file():
            destino = pasta / arquivo.suffix.lstrip(".").lower()
            destino.mkdir(exist_ok=True)
            shutil.move(str(arquivo), str(destino / arquivo.name))

if __name__ == "__main__":
    organizar(sys.argv[1])
'''
(REPO / "organiza.py").write_text(ORGANIZA_V1, encoding="utf-8")

print("oficina montada em:", BASE)
print("versão do git:", subprocess.run(["git","--version"],capture_output=True,text=True).stdout.strip())

---
# 1. A dívida: *"mudei alguma coisa e não sei o quê"*

Sábado passado o script passou a rodar sozinho. E aí veio o problema de sempre: você mexeu, ficou
pior, e não sabe voltar. A solução que todo mundo inventa sozinho é esta:

In [ ]:
for p in sorted(HORROR.iterdir()):
    print(" ", p.name)

Olhe de verdade essa lista e responda três perguntas:

1. Qual é o mais novo?
2. O que mudou entre o `organiza_v2.py` e o `organiza_v2_ok.py`?
3. Por que você fez essa mudança?

Você não sabe responder nenhuma — e esse arquivo é **seu**, de **ontem**.

É isso que o git resolve. Não é ferramenta de equipe grande; é **memória** para uma pessoa só.

### Os três lugares onde seu código mora

Essa é a única teoria da aula. Decore estes três nomes e 90% da confusão com git vai embora:

| Lugar | O que é | Comando que leva para o próximo |
|---|---|---|
| **Pasta de trabalho** | os arquivos como estão agora na sua tela | `git add` |
| **Área de preparo** *(staging)* | o rascunho do próximo commit — o que você já escolheu | `git commit` |
| **Histórico** | a fila de commits, cada um com código, autor, data e mensagem | — |

`git status` é a pergunta *"onde está cada coisa agora?"*. Você vai rodar isso o tempo todo.

---
# 2. `git init` e `git status`

`git init` cria uma pasta escondida `.git` dentro do projeto. **É só isso.** O repositório não está
em nuvem nenhuma — está num diretório no seu computador.

O `-b main` diz que o nome do primeiro ramo é `main` (o GitHub usa esse nome; o git antigo usava
`master` e ainda avisa sobre isso se você não disser nada).

In [ ]:
git("init", "-b", "main")
git("status")

Leia a saída do `status` em voz alta. Ele diz três coisas:

- `On branch main` — em que ramo você está
- `No commits yet` — o histórico está vazio
- `Untracked files: organiza.py` — **o git viu o arquivo e está de fora dele**

> *Untracked* é a palavra mais importante dessa saída. O git **não** versiona nada sozinho.
> Ele versiona o que você mandar.

### `git add`: escolher o que entra no próximo commit

In [ ]:
git("add", "organiza.py")
git("status", "--short")

O `--short` é a versão resumida — você vai usar mais que a longa. A legenda:

| Marca | Significa |
|---|---|
| `??` | o git não conhece esse arquivo (*untracked*) |
| `A` | arquivo novo, já na área de preparo |
| `M` na 2ª coluna | modificado, **ainda não** preparado |
| `M` na 1ª coluna | modificado **e** preparado |

### `git commit`: fechar o pacote

O `-m` é a mensagem. Guarde isso desde o primeiro commit: **a mensagem responde *por quê*;
o diff já mostra *o quê*.** Vamos voltar nisso no bloco 6, que é onde mais gente erra.

In [ ]:
git("commit", "-m", "organizador move arquivo para subpasta por extensao")
git("log")

Aquele monte de letras e números é o **hash** do commit — o nome dele. Sete caracteres do começo já
bastam para você se referir a ele.

> Esse commit é um **ponto de retorno**. De agora em diante, aconteça o que acontecer com o
> `organiza.py`, esta versão existe e é recuperável. É daqui que vem a coragem de mexer no código.

---
# 3. `git diff`: o que exatamente eu mudei

Agora vamos mexer no script: entra o **modo simulação** da Aula 13 — aquele que só mostra o que
faria, sem mover arquivo nenhum.

In [ ]:
ORGANIZA_V2 = ORGANIZA_V1.replace(
    "def organizar(pasta):",
    "def organizar(pasta, simular=False):"
).replace(
    '            destino.mkdir(exist_ok=True)',
    '            if simular:\n'
    '                print(f"moveria {arquivo.name} -> {destino.name}/")\n'
    '                continue\n'
    '            destino.mkdir(exist_ok=True)'
)
(REPO / "organiza.py").write_text(ORGANIZA_V2, encoding="utf-8")

git("status", "--short")
git("diff")

Esta é a saída que paga a dívida da aula. Como ler:

- `---` é como estava, `+++` é como está
- `@@ -1,11 +1,14 @@` — mexeu no trecho que começa na linha 1; eram 11 linhas, agora são 14
- linha com `-` na frente: **saiu**. Linha com `+`: **entrou**. Linha sem nada: só contexto

> *"Mudei alguma coisa e não sei o quê"* — **esta** é a resposta. Quatro linhas entraram, uma saiu,
> e você está olhando exatamente para elas.

Agora o commit da mudança. Repare na mensagem: ela diz **por que** o modo simulação existe.

In [ ]:
git("add", "organiza.py")
git("commit", "-m", "modo simulacao: testar sem mover arquivo nenhum")
git("log", "--oneline")

`git log --oneline` é o comando que você mais vai usar na vida. Um commit por linha.

**A regra de ouro do diff:** rode `git diff` **antes** de todo `git add`. Leva dois segundos e é a
última chance de ver o que você está guardando — inclusive aquele `print("teste 123")` que você
esqueceu no meio do código.

---
# 4. `.gitignore`: o que **não** deve entrar

Vamos simular o que aparece na sua pasta depois de rodar o projeto algumas vezes: o cache do Python,
o ambiente virtual (o `venv` que a Aula 11 prometeu) e o arquivo de log da Aula 13.

In [ ]:
(REPO / "__pycache__").mkdir(exist_ok=True)
(REPO / "__pycache__" / "organiza.cpython-311.pyc").write_bytes(b"\x00\x01lixo binario")
(REPO / ".venv").mkdir(exist_ok=True)
(REPO / ".venv" / "pyvenv.cfg").write_text("home = /usr/bin\n", encoding="utf-8")
(REPO / "organiza.log").write_text("2026-11-14 10:31:02 INFO 7 arquivos movidos\n", encoding="utf-8")

git("status", "--short")

Três coisas ali que **não são o seu código**:

- `__pycache__/` — o Python gera de novo sozinho, e é binário (o diff fica ilegível)
- `.venv/` — pode ter **milhares** de arquivos e é específico da sua máquina; quem clona recria com
  o `requirements.txt` da Aula 11
- `*.log` — cresce para sempre e muda a cada execução; cada commit viraria um diff gigante de lixo

A solução é um arquivo de texto chamado `.gitignore`, uma regra por linha:

In [ ]:
(REPO / ".gitignore").write_text(
    "__pycache__/\n"
    ".venv/\n"
    "*.log\n",
    encoding="utf-8")

git("status", "--short")

Sumiram. Sobrou só o `.gitignore` em si — que **entra** no commit, porque a regra também é parte do
projeto: quem clonar herda os mesmos "não commite isso".

> Se for gerado, baixado, pesado ou secreto: **não vai para o git.**

In [ ]:
git("add", ".gitignore")
git("commit", "-m", "ignora cache, venv e log: nada disso e meu codigo")

---
# 5. 💥 Quebrar de propósito nº 1 — **o git lembra para sempre**

**Este é o bloco mais importante da aula.** Não pule, não leia por cima, e rode cada célula.

A história: você resolveu fazer o organizador avisar no Telegram quando terminar. Para isso precisa
de uma chave. Você cria um `config.py`, cola a chave lá, e commita junto porque senão o programa
não roda.

In [ ]:
(REPO / "config.py").write_text(
    'CHAVE_API = "xgh-live-8f3b91d04c7a4e26"\n'
    'CHAT_ID = "-1001998877665"\n',
    encoding="utf-8")

(REPO / "avisa.py").write_text('''import schedule, time
from config import CHAVE_API, CHAT_ID
from organiza import organizar

def tarefa():
    organizar("/home/turma/Downloads")

schedule.every().day.at("03:00").do(tarefa)
while True:
    schedule.run_pending()
    time.sleep(60)
''', encoding="utf-8")

git("add", ".")
git("commit", "-m", "avisa no telegram quando terminar")

Duas semanas depois cai a ficha: **a chave está no repositório**. Você faz exatamente o que qualquer
pessoa faria — tira o arquivo, coloca no `.gitignore`, deixa um exemplo sem segredo no lugar, e
commita o conserto.

`git rm --cached` significa *"pare de versionar isso, mas não apague do meu disco"*.

In [ ]:
git("rm", "--cached", "config.py")

(REPO / ".gitignore").write_text(
    "__pycache__/\n"
    ".venv/\n"
    "*.log\n"
    "config.py\n",
    encoding="utf-8")

(REPO / "config.exemplo.py").write_text(
    'CHAVE_API = "cole-a-sua-chave-aqui"\n'
    'CHAT_ID = "cole-o-seu-chat-id"\n',
    encoding="utf-8")

os.remove(REPO / "config.py")

git("add", ".")
git("commit", "-m", "tira a chave do repositorio")

### E agora parece resolvido

Olhe a pasta e olhe o `status`. O `config.py` não está mais lá, e o git diz *nothing to commit,
working tree clean*. É neste ponto que todo mundo respira aliviado e vai dormir.

*(O `.venv`, o `__pycache__` e o `organiza.log` continuam na pasta — eles estão **ignorados**, não
apagados. Ignorar é dizer ao git para não olhar, não ao sistema para deletar.)*

In [ ]:
print("$ ls")
print(" ", "  ".join(sorted(p.name for p in REPO.iterdir() if p.name != ".git")))
print()
git("status")
git("log", "--oneline")

### 💥 Agora o soco

O commit **não é um save**. É uma **foto**. Apagar o arquivo hoje não apaga a foto de ontem.

Rode a célula abaixo. Ela não tem truque nenhum: é um comando de git que qualquer pessoa com acesso
ao repositório pode rodar.

In [ ]:
inicio = time.perf_counter()
historico = git_quieto("log", "-p", "--", "config.py")
print(historico)
segundos = time.perf_counter() - inicio

print("-" * 60)
print(f"tempo para recuperar a chave apagada: {segundos:.3f} segundos")
print("a chave aparece no histórico:", "xgh-live-8f3b91d04c7a4e26" in historico)

Leia o que apareceu. **Dois** commits mexeram no `config.py`:

- o commit em que ela **entrou** (`+CHAVE_API = ...`)
- e o commit em que você a **apagou** (`-CHAVE_API = ...`) — que, para funcionar como diff,
  **precisa dizer o que foi apagado**

Ou seja: o commit do conserto é o que mais grita a chave. Você a escreveu no histórico **duas vezes**.

E não para aí. Dá para pedir o arquivo inteiro, como ele era naquele dia:

In [ ]:
commits_do_config = git_quieto("rev-list", "--all", "--", "config.py").split()
primeiro = commits_do_config[-1][:7]
print("o commit que criou o config.py:", primeiro)
print()
git("show", f"{primeiro}:config.py")

E dá para varrer **todos** os commits do repositório procurando um pedaço de texto. É literalmente o
que um robô faz, 24 horas por dia, em cima de tudo que é público no GitHub:

In [ ]:
todos_os_commits = git_quieto("rev-list", "--all").split()
print(f"procurando xgh-live em {len(todos_os_commits)} commits...")
print()
achado = git_quieto("grep", "-n", "xgh-live", *todos_os_commits)
print("$ git grep -n xgh-live $(git rev-list --all)")
print(achado)

### O que fazer de verdade quando isso acontece

Ordem importa. A primeira linha é a única que resolve:

1. **Revogue a chave.** Entre no painel do provedor e invalide. A chave vazada não tem conserto —
   ela tem substituição. Tudo o que vem depois é limpeza.
2. Gere uma nova e guarde **fora** do repositório (`config.py` ignorado, ou variável de ambiente).
3. Commite o `.gitignore` e um `config.exemplo.py` sem segredo, para quem clonar saber o que criar.
4. **Só então** pense em reescrever o histórico (`git filter-repo`, `BFG`). É cirurgia: muda todos os
   hashes depois do ponto, quebra o repositório de quem já clonou, e **não serve para nada** se a
   chave já foi copiada. É por isso que o passo 1 é o passo 1.

### 🌱 A semente da Aula 18

Hoje a chave é falsa e o prejuízo é zero. **Na Aula 18 vocês vão ter uma chave de LLM de verdade na
mão** — uma que chama modelo de linguagem, e que, se alguém pegar, gasta na conta de vocês.

É por isso que esta aula vem antes daquela, e não depois. Quando chegarmos lá, a primeira coisa que
eu vou perguntar é: *"onde você guardou a chave?"*

> O segredo mais seguro do mundo é o que **nunca** entrou num commit.

---
# 6. 💥 Quebrar de propósito nº 2 — mensagem de commit inútil

A célula abaixo monta **dois repositórios com exatamente as mesmas mudanças de código**. A única
diferença é o texto das mensagens. Depois imprime os dois `git log --oneline` lado a lado.

In [ ]:
MENSAGENS_RUINS = ["ajustes", "teste", "agora vai", "update", "correcoes", "fix"]

MENSAGENS_BOAS = [
    "organizador move arquivo para subpasta por extensao",
    "modo simulacao: testar sem mover arquivo nenhum",
    "ignora cache, venv e log",
    "log em arquivo: script das 3h tem que deixar rastro",
    "caminho absoluto: o relativo quebrava no agendador",
    "nao refaz o que ja foi feito: rodar duas vezes nao duplica",
]

logs = []
for nome, mensagens in [("ruim", MENSAGENS_RUINS), ("bom", MENSAGENS_BOAS)]:
    pasta = BASE / nome
    pasta.mkdir(exist_ok=True)
    git_quieto("init", "-b", "main", onde=pasta)
    for i, m in enumerate(mensagens):
        (pasta / "organiza.py").write_text(f"# versao {i}\n" * (i + 1), encoding="utf-8")
        git_quieto("add", ".", onde=pasta)
        git_quieto("commit", "-m", m, onde=pasta)
    logs.append(git_quieto("log", "--oneline", onde=pasta).splitlines())

esquerda, direita = logs
ESQ = "git log --oneline    RUIM"
DIR = "git log --oneline    BOM"
largura = max([len(x) for x in esquerda] + [len(ESQ)]) + 3
print(ESQ.ljust(largura) + "| " + DIR)
print("-" * largura + "+" + "-" * 62)
for a, b in zip(esquerda, direita):
    print(a.ljust(largura) + "| " + b)

Olhe a coluna da esquerda e tente responder: **em qual desses commits entrou o modo simulação?**

Não dá. E não é que a informação esteja escondida — ela **nunca foi escrita**. O git guardou o código
perfeitamente; o que se perdeu foi o *porquê*, que só existia na sua cabeça naquela terça-feira.

Agora a coluna da direita. Ela é um **diário do projeto**. Dá para ler de cima a baixo e entender a
história: nasceu movendo arquivo, ganhou modo de teste, parou de versionar lixo, ganhou log,
consertou o caminho que quebrava no agendador, e virou idempotente.

### A regra

| | |
|---|---|
| **O quê** mudou | o `diff` já mostra — não repita |
| **Por quê** mudou | só você sabe — **é isso que vai na mensagem** |

Um teste rápido que funciona: a mensagem tem que fazer sentido depois de *"este commit..."*.

- *"este commit **ajustes**"* — não é frase.
- *"este commit **faz o caminho ser absoluto porque o relativo quebrava no agendador**"* — é.

E se você não consegue resumir numa linha, normalmente o problema não é a frase: **é que o commit
está fazendo cinco coisas ao mesmo tempo.** Commit pequeno é mensagem fácil.

---
# 7. 💥 Quebrar de propósito nº 3 — clonar e não rodar

`git clone` copia um repositório inteiro, com histórico e tudo. Vamos clonar o nosso para outra
pasta, como se fosse o seu colega baixando do GitHub.

In [ ]:
CLONE = BASE / "clone-do-colega"
git("clone", str(REPO), str(CLONE), onde=BASE)

print("$ ls clone-do-colega/")
print(" ", "  ".join(sorted(p.name for p in CLONE.iterdir() if p.name != ".git")))

Agora o colega tenta rodar. Esta célula **vai dar erro** — leia a última linha:

In [ ]:
resultado = subprocess.run([sys.executable, "avisa.py"],
                           capture_output=True, text=True, cwd=str(CLONE))
print("$ python avisa.py")
print(resultado.stdout + resultado.stderr)

`ModuleNotFoundError: No module named 'schedule'` — o `ModuleNotFoundError` da Aula 11, de volta e
mais caro: agora é na máquina de **outra pessoa**, que não sabe o que instalar.

Falta o `requirements.txt`. Conserto em duas linhas:

In [ ]:
(REPO / "requirements.txt").write_text("schedule==1.2.2\n", encoding="utf-8")
git("add", "requirements.txt")
git("commit", "-m", "requirements.txt: sem isso o projeto nao roda na maquina de ninguem")

E tem um **segundo** motivo para não rodar, que é consequência direta do bloco 5: o `config.py` está
no `.gitignore`, então ele **não veio no clone** — e isso está certo. Mas o `avisa.py` importa dele.

A célula abaixo mostra o erro de dentro do Python mesmo. **Também vai dar erro, de propósito:**

In [ ]:
sys.path.insert(0, str(CLONE))
from config import CHAVE_API

`ModuleNotFoundError: No module named 'config'`.

Esses dois erros são o teste de qualidade do Projeto 2, e dão a lista do que o `README.md` precisa
responder:

| O que falta | O que resolve |
|---|---|
| as bibliotecas | `requirements.txt` + uma linha `pip install -r requirements.txt` no README |
| o arquivo de segredo | `config.exemplo.py` versionado + o README dizendo *"copie para `config.py` e preencha"* |
| saber o que o programa é | a primeira frase do README |
| saber como rodar | a linha de comando exata, copiável |

> **O teste do seu README:** entregue o link do repositório para alguém que nunca viu o projeto e
> fique calado. Se a pessoa consegue rodar, o README presta. Se ela te pergunta algo, a resposta
> dela é a linha que está faltando.

---
# 8. GitHub: `remote` e `push`

Até aqui **nada saiu da sua máquina**. O git é local; o GitHub é um lugar onde você guarda uma cópia.

| | |
|---|---|
| **git** | o programa que guarda o histórico, no seu computador |
| **GitHub** | um site que hospeda repositórios git (tem GitLab, Bitbucket, Codeberg...) |

### 8.1 — Configurar quem você é (uma vez na vida, por máquina)

```
git config --global user.name "Seu Nome"
git config --global user.email "seu@email.com"
```

Use o **mesmo e-mail da conta do GitHub** — é assim que os commits aparecem com a sua cara no site.

### 8.2 — Criar o repositório vazio no site

No GitHub: botão **New** → nome do repositório → **Public** → **não** marque nada das caixinhas
(`README`, `.gitignore`, licença). Repositório vazio evita conflito no primeiro `push`.

### 8.3 — Ligar o seu repositório local ao do site

```
git remote add origin https://github.com/SEU-USUARIO/organizador.git
git remote -v
git push -u origin main
```

- `origin` é só um apelido para o endereço. Poderia ser qualquer nome; todo mundo usa `origin`.
- `-u origin main` liga o seu `main` local ao `main` de lá. **Da segunda vez em diante é só `git push`.**

### 8.4 — A senha que não é senha

O GitHub **não aceita** mais a senha da conta no `git push`. Você vai usar um **Personal Access Token**:

`github.com` → foto → *Settings* → *Developer settings* → *Personal access tokens* → *Tokens (classic)*
→ *Generate new token* → marque o escopo **`repo`** → copie.

Quando o `git push` pedir usuário e senha: usuário é o seu login, **senha é o token**.

> Esse token é exatamente o tipo de coisa do bloco 5. Ele dá acesso de escrita aos seus repositórios.
> **Nunca** em arquivo, nunca em commit, nunca no grupo do WhatsApp.

*(Se preferir, a extensão GitHub do VS Code faz o login pelo navegador e você não vê token nenhum.
Mais confortável — mas saiba que o token existe, para entender o erro quando aparecer.)*

### 8.5 — O ciclo do dia a dia

```
git status      # onde está cada coisa
git diff        # o que exatamente eu mudei
git add .       # escolho o que entra
git commit -m "por que eu mudei"
git push        # manda para o GitHub
```

Cinco comandos. É isso que 95% do uso de git é, para sempre.

---
# 9. Um `README.md` que presta

O README é a **capa** do seu projeto. Quem abre o repositório vê ele antes do código — e, se ele for
ruim, não vê o código nenhum.

Cinco seções resolvem. Nessa ordem, porque é a ordem das perguntas de quem chega:

In [ ]:
README = """# organizador

Organiza uma pasta movendo cada arquivo para uma subpasta com o nome da extensão.
Feito para rodar sozinho na pasta de Downloads, de madrugada.

## Como instalar

```
git clone https://github.com/SEU-USUARIO/organizador.git
cd organizador
python -m venv .venv
pip install -r requirements.txt
```

Copie `config.exemplo.py` para `config.py` e preencha com a sua chave.
`config.py` está no `.gitignore` de propósito: segredo nao vai para o git.

## Como rodar

```
python organiza.py ~/Downloads          # organiza de verdade
python organiza.py ~/Downloads --simular # so mostra o que faria
```

## Como agendar

Windows: Agendador de Tarefas, acao "iniciar programa", com o caminho **absoluto**
do python e do script. Linux/Mac: `crontab -e`.

## Detalhes

- Rodar duas vezes nao duplica nada: se o arquivo ja esta na subpasta certa, nao mexe.
- Escreve em `organiza.log` o que fez, com data e hora.
- Python 3.10 ou mais novo.
"""

(REPO / "README.md").write_text(README, encoding="utf-8")
git("add", "README.md")
git("commit", "-m", "readme: o que e, como instalar, como rodar e como agendar")
git("log", "--oneline")

Esse é o histórico de um projeto que dá para entregar. Oito commits, cada um com um motivo legível.

**Três coisas que fazem um README ruim ficar bom na hora:**

1. A **primeira frase** diz o que o programa faz, sem rodeio e sem "este projeto tem como objetivo".
2. Os comandos estão em bloco de código, **copiáveis e completos** — não "instale as dependências".
3. Tem a linha do caso chato: o segredo que precisa ser criado, a versão do Python, o agendamento.

---
# 10. 🛠️ Projeto 2 — critérios de aceite

**Entrega: Aula 16, em duas semanas.** O que você entrega é **um link de repositório público**.

É a automação da Aula 13 (ou outra sua, se preferir) publicada no GitHub.
Sugestões, se a sua não estiver de pé: organizador de Downloads · relatório automático de uma
planilha sua · monitor de cotação ou de preço · gerador de recibo em PDF.

## O quadro

| # | Critério | Como eu vou conferir |
|---|---|---|
| 1 | **Roda sozinho** — é um `.py`, recebe o que precisa por argumento, não pede nada no meio | eu rodo pelo terminal, sem abrir o código |
| 2 | **`README.md`** com: o que é · como instalar · como rodar (comando copiável) | eu leio o README e tento usar **sem te perguntar nada** |
| 3 | **`requirements.txt`** | eu clono num ambiente limpo e instalo com `pip install -r` |
| 4 | **`.gitignore`** — sem `__pycache__/`, sem `.venv/`, sem log, **sem segredo** | eu abro a lista de arquivos do repositório |
| 5 | **Sobrevive a rodar duas vezes seguidas sem duplicar nada** | eu rodo duas vezes e comparo o resultado |
| 6 | **Histórico legível** — no mínimo 5 commits com mensagem que diz *por quê* | `git log --oneline` |

### O critério 5 é o que mais reprova

É a dívida da Aula 13 cobrada com juros. Antes de entregar, faça isto e olhe o resultado:

```
python seu_script.py <pasta-de-teste>
python seu_script.py <pasta-de-teste>
```

Duas vezes seguidas, o mesmo comando. Se na segunda vez apareceu linha repetida no relatório,
arquivo duplicado, e-mail mandado duas vezes ou erro — **não está pronto**. As duas saídas:
**verificar antes de agir** ("o arquivo já está lá? então não mexe") ou **marcar o que já foi feito**.

### Atalho honesto

Se a automação da Aula 13 não está de pé, **publique o que existe**, com um README dizendo o que falta.
Repositório com projeto pequeno e honesto vale mais que repositório vazio com nome bonito.

---
# 11. O que isso destrava

- **Mexer no código sem medo.** Quebrou? Volta. O commit é o ponto de retorno.
- **Trabalhar na mesma coisa de dois computadores** — o do trabalho e o de casa.
- **Ter o que mostrar.** Link de repositório é o portfólio de quem programa. Não é currículo falando
  de você; é código seu, com data.
- **Pegar código do mundo.** `git clone` de qualquer projeto público — você já pode ler, rodar e mexer
  em projeto de gente que você nunca vai conhecer.
- **Guardar segredo no lugar certo**, que é o que a Aula 18 vai exigir de vocês.

> Você não ficou melhor em Python hoje. Você ficou com **direito de errar**.

---
# 12. 📝 Tarefa da semana

**1. Publique um repositório.** Qualquer projeto seu — pode ser o Projeto 1. Com `README.md`,
`.gitignore` e `requirements.txt`. **Mande o link no grupo.**

**2. Cinco commits de verdade**, cada um com mensagem que passa no teste do *"este commit..."*.
Commits pequenos, um motivo cada.

**3. Abra o repositório de um colega e tente rodar sem perguntar nada a ele.** Depois mande para a
pessoa, no privado, **a primeira coisa que te travou**. Isso vale mais que elogio.

**4. Rode o seu script duas vezes seguidas** e diga no grupo o que aconteceu na segunda. Se duplicou,
ótimo: achou o que consertar antes da entrega.

**5. Desafio.** Procure na documentação do git o que `git restore` e `git revert` fazem, e descubra
a diferença entre os dois. É a Aula 11 aplicada: ler documentação de uma ferramenta que ninguém te
ensinou. Traga a resposta em uma frase.

---

## A pergunta que fica

Olhe para tudo que a gente fez em quatorze aulas. **Cada decisão que o seu programa toma, foi você
quem escreveu** — `if nota >= 7`, `if extensao == ".pdf"`, `if status_code == 200`.

O limiar é seu. A regra é sua.

**E se eu não souber a regra?** Se eu tiver dez mil e-mails, metade spam, e ninguém no mundo
conseguir escrever o `if` que separa os dois?

**Como é que o programa decide a partir do dado, e não da regra que eu escrevi?**

*(Aula 15: machine learning de verdade — e a gente volta na Aula 3.)*

---

*Mestres do XGH · Aula 14 · 14/11/2026*

In [ ]:
# Seu espaço — testes, anotações, o que você quiser
# A oficina de hoje está em:
print(BASE)